# 보고서·PPT용 자료 일괄 생성 (오케스트레이터)

**역할:** PDF/PPT에 넣을 **표·그림을 한 번에** 다시 뽑을 때 쓴다.  
**원칙:** 학습·벤치마크 로직은 `fds_pipeline.py`·`scripts/*.py`에 두고, 이 노트북은 **같은 스크립트를 순서대로 실행**만 한다 (코드 중복 없음).

**전제:** 프로젝트 루트(`personalProject`)에서 이 노트북을 연다. `dataset/creditcard.csv` 필요.

**수동:** `fds_report_figures.ipynb`는 여기 포함하지 않았다 (노트북이 길고 셀 단위 편집용). 보고용 번호 PNG는 그쪽 실행 후 `report_figures/` 루트를 확인한다.

**시간:** SMOTETomek·RandomizedSearchCV 때문에 **수십 분** 걸릴 수 있다.

In [1]:
from pathlib import Path
import os

# 노트북 위치와 무관하게 프로젝트 루트로 맞춤 (report_generate_all.ipynb 가 루트에 있을 때)
ROOT = Path.cwd()
if not (ROOT / "fds_pipeline.py").is_file():
    raise SystemExit("fds_pipeline.py 가 보이는 폴더(프로젝트 루트)에서 노트북을 여세요.")
os.chdir(ROOT)
print("cwd:", ROOT.resolve())

cwd: D:\personalProject


## 1) 아티팩트 `artifacts/*.pkl`

In [2]:
!python scripts/train_save_artifacts.py

저장 완료:
  artifacts\fds_model.pkl (351 KB)
  artifacts\fds_scaler.pkl (0 KB)
  artifacts\fds_X_test.pkl (14241 KB)
  artifacts\fds_y_test.pkl (1780 KB)


## 2) EDA·보조 플롯 → `report_figures/eda/`

In [3]:
!python scripts/report_plots/plot_eda_class_distribution.py
!python scripts/report_plots/plot_eda_correlation_heatmap.py
!python scripts/report_plots/plot_eda_time_kde.py
!python scripts/report_plots/plot_eda_amount.py
!python scripts/report_plots/plot_preprocess_time_sin_cos.py
!python scripts/report_plots/plot_ch8_descriptive_statistics.py

저장: D:\personalProject\report_figures\eda\eda_class_distribution.png
저장: D:\personalProject\report_figures\eda\eda_correlation_heatmap.png
저장: D:\personalProject\report_figures\eda\eda_time_kde.png
저장: D:\personalProject\report_figures\eda\eda_amount_box_hist.png
저장: D:\personalProject\report_figures\eda\preprocess_time_sin_cos_kde.png
CSV(전체 변수): D:\personalProject\report_figures\eda\ch8_descriptive_statistics.csv
CSV(Class별 T/A): D:\personalProject\report_figures\eda\ch8_descriptive_by_class_Time_Amount.csv
PNG(슬라이드용): D:\personalProject\report_figures\eda\ch8_descriptive_compact.png


## 3) 1차 벤치마크 + 시각화 → `report_figures/` (루트)

In [4]:
!python scripts/model_comparison.py
!python scripts/model_comparison_visualize.py --out-dir report_figures


=== Hold-out Test (threshold=0.5), Train=SMOTETomek / Test=원 불균형 ===

                            model   recall       f1       f2   pr_auc  cv_best_f2
    LogisticRegression (baseline) 0.918367 0.108173 0.229826 0.719884         NaN
LogisticRegression (tuned, CV=F2) 0.918367 0.108043 0.229592 0.721109    0.936532
 RandomForest (baseline, 2-Track) 0.816327 0.851064 0.829876 0.872667         NaN
 XGBoost (baseline, fds defaults) 0.857143 0.597865 0.730435 0.851009         NaN
           XGBoost (tuned, CV=F2) 0.816327 0.812183 0.814664 0.865765    0.999935

=== Tuned best params (요약) ===

LogisticRegression: {'lr__C': np.float64(6.79657809075816), 'lr__class_weight': 'balanced', 'lr__solver': 'lbfgs'}
XGBoost: {'xgb__n_estimators': 251, 'xgb__max_depth': 7, 'xgb__learning_rate': np.float64(0.14701460042617717), 'xgb__subsample': np.float64(0.916163822772898), 'xgb__colsample_bytree': np.float64(0.76124547565948), 'xgb__min_child_weight': 3, 'xgb__reg_lambda': np.float64(0.3550012525851

## 4) 2차 + val F2 곡선

In [5]:
!python scripts/model_comparison_round2.py
!python scripts/report_plots/plot_round2_val_f2_vs_threshold.py

[1/3] RandomForest baseline...
[2/3] XGBoost baseline...
[3/3] XGBoost RandomizedSearchCV (F2)...

=== Round 2: val에서 F2 최대 임계값 → test 지표 ===

설정: val_fraction=0.15, random_state=42

                           model  val_best_thr_f2  val_f2_at_thr  cv_best_f2_train   recall       f1       f2   pr_auc  roc_auc
RandomForest (baseline, 2-Track)             0.45       0.795848               NaN 0.826531 0.843750 0.833333 0.866707 0.968788
XGBoost (baseline, fds defaults)             0.87       0.798611               NaN 0.816327 0.816327 0.816327 0.819797 0.978213
          XGBoost (tuned, CV=F2)             0.84       0.804196          0.999934 0.826531 0.875676 0.845511 0.868770 0.979424

=== XGB tuned best params (요약) ===

{'xgb__n_estimators': 251, 'xgb__max_depth': 7, 'xgb__learning_rate': np.float64(0.14701460042617717), 'xgb__subsample': np.float64(0.916163822772898), 'xgb__colsample_bytree': np.float64(0.76124547565948), 'xgb__min_child_weight': 3, 'xgb__reg_lambda': np.float64(0.3

## 5) 도식·학습곡선

In [6]:
!python report/generate_report_model_diagrams.py

저장: report_figures\model_diagram_two_track.png
저장: report_figures\model_diagram_bagging_boosting.png
저장: report_figures\model_diagram_learning_curve.png
완료.


## 6) (선택) extras / 전처리 민감도

In [7]:
!python report/generate_report_extras.py
!python scripts/compare_preprocessing_strategies.py

저장: report_figures\07_top_v_kde_by_class.png
저장: report_figures\08_smotetomek_pca_before_after.png
완료.
[실험 1 - 1/2] SMOTETomek + 학습: RobustScaler full df fit (현재 파이프라인)...
[실험 1 - 2/2] SMOTETomek + 학습: RobustScaler train-only fit...
[실험 2 - 1/2] SMOTETomek + 학습...
[실험 2 - 2/2] scale_pos_weight만 학습...

=== 동일 split (stratify, random_state=42, test_size=0.2), 동일 XGB 베이스라인 ===

Train/Test 건수: Train=227845 (사기 394) | Test=56962 (사기 98)

[표] 실험 1: Amount 스케일만 다름. 둘 다 Train에 SMOTETomek 후 학습

                             스케일 방식   recall       f1       f2   pr_auc
RobustScaler full df fit (현재 파이프라인) 0.857143 0.597865 0.730435 0.851009
        RobustScaler train-only fit 0.857143 0.600000 0.731707 0.824311

[표] 실험 2: RobustScaler는 full df fit로 통일. 불균형 처리만 다름

                                      불균형 처리   recall       f1       f2   pr_auc
       SMOTETomek (train 사기 394건 → 리샘플 후 학습) 0.857143 0.597865 0.730435 0.851009
리샘플 없음 + scale_pos_weight=577.2868 (neg/pos) 0.836735 0.811881 0.826613 0.864

## 7) PPT·PDF에 붙일 **표** 미리보기 (CSV)

In [8]:
import pandas as pd
from pathlib import Path

p1 = Path("report_figures/model_compare_all_metrics.csv")
p2 = Path("report_figures/model_round2_val_thr.csv")
if p1.is_file():
    display(pd.read_csv(p1))
else:
    print("없음:", p1)
if p2.is_file():
    display(pd.read_csv(p2))
else:
    print("없음:", p2)

,model,cv_best_f2_train,recall,f1,f2,pr_auc,roc_auc
0,로지스틱 회귀 (baseline),NaN,0.918367,0.108173,0.229826,0.719884,0.973026
1,"로지스틱 회귀 (tuned, CV=F2)",0.936532,0.918367,0.108043,0.229592,0.721109,0.972904
2,랜덤 포레스트 (baseline),NaN,0.816327,0.851064,0.829876,0.872667,0.972507
3,XGBoost (baseline),NaN,0.857143,0.597865,0.730435,0.851009,0.974444
4,"XGBoost (tuned, CV=F2)",0.999935,0.816327,0.812183,0.814664,0.865765,0.980858


,model,val_best_thr_f2,val_f2_at_thr,cv_best_f2_train,recall,f1,f2,pr_auc,roc_auc
0,"RandomForest (baseline, 2-Track)",0.45,0.795848,NaN,0.826531,0.843750,0.833333,0.866707,0.968788
1,"XGBoost (baseline, fds defaults)",0.87,0.798611,NaN,0.816327,0.816327,0.816327,0.819797,0.978213
2,"XGBoost (tuned, CV=F2)",0.84,0.804196,0.999934,0.826531,0.875676,0.845511,0.868770,0.979424
